In [1]:
from pathlib import Path
import json
import html


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

JSON_INPUT_FILE = (
    PROCESSED_DATA_DIR
    / "steam_games_2022_2025_compact.json"
)

APP_OUTPUT_DIR = (
    PROJECT_ROOT
    / "steam-game-suggester"
)

APP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HTML_OUTPUT_FILE = (
    APP_OUTPUT_DIR
    / "steam_game_suggester.html"
)


# ============================================================
# LOAD DATA
# ============================================================

with open(
    JSON_INPUT_FILE,
    "r",
    encoding="utf-8"
) as file:
    games_data = json.load(file)


if not isinstance(games_data, list):
    raise TypeError(
        "Expected a list of game records."
    )

if not games_data:
    raise ValueError(
        "The compact game JSON is empty."
    )


# ============================================================
# VALIDATE FIELDS
# ============================================================

required_fields = {
    "appid",
    "name",
    "release_year",
    "price",
    "total_reviews",
    "review_score",
    "estimated_revenue",
    "revenue_rank_in_year",
    "qualifying_games_in_year",
    "revenue_percentile_in_year",
}

available_fields = set()

for game in games_data[:200]:
    available_fields.update(game.keys())

missing_fields = (
    required_fields
    - available_fields
)

if missing_fields:
    raise ValueError(
        "Missing fields: "
        + ", ".join(sorted(missing_fields))
    )


# ============================================================
# DERIVED GENERATOR DATA
# ============================================================

years = sorted({
    int(game["release_year"])
    for game in games_data
})

maximum_price = max(
    float(game.get("price", 0) or 0)
    for game in games_data
)

maximum_rank = max(
    int(
        game.get(
            "revenue_rank_in_year",
            1
        ) or 1
    )
    for game in games_data
)

all_tags = sorted({
    str(tag).strip()
    for game in games_data

    for tag in (
        game.get("tags", [])
        if isinstance(
            game.get("tags", []),
            list
        )
        else (
            game.get("tags", {}).keys()
            if isinstance(
                game.get("tags", {}),
                dict
            )
            else []
        )
    )

    if str(tag).strip()
})


# ============================================================
# EMBED GAME JSON
# ============================================================

embedded_json = json.dumps(
    games_data,
    ensure_ascii=False,
    separators=(",", ":")
)

# Avoid accidentally terminating the HTML script block
embedded_json = embedded_json.replace(
    "</script>",
    "<\\/script>"
)


# ============================================================
# YEAR CONTROLS
# ============================================================

year_checkbox_html = "\n".join(
    f"""
    <label class="year-option">
        <input
            type="checkbox"
            class="year-checkbox"
            value="{year}"
            checked
        >
        <span>{year}</span>
    </label>
    """
    for year in years
)


# ============================================================
# TAG CONTROLS
# ============================================================

tag_buttons_html = "\n".join(
    (
        f'<button '
        f'class="available-tag" '
        f'type="button" '
        f'data-tag="{html.escape(tag, quote=True)}">'
        f'{html.escape(tag)}'
        f'</button>'
    )
    for tag in all_tags
)


# ============================================================
# HTML TEMPLATE
#
# Important:
# This is NOT a Python f-string.
#
# Therefore JavaScript/CSS can use normal:
#
#     {
#     }
#
# instead of:
#
#     {{
#     }}
#
# ============================================================

html_document = r"""
<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>Indie Case Study Suggester</title>


<style>

:root {
    color-scheme: dark;

    --bg: #080c13;
    --panel: #111823;
    --panel-2: #151e2b;

    --border: #27344a;
    --border-soft: #1e2a3c;

    --text: #f4f7fc;
    --muted: #9aabc1;
    --subtle: #718198;

    --accent: #63c2ff;
    --accent-strong: #25a9ff;
    --accent-soft: rgba(37,169,255,0.12);

    --green: #66d492;
    --green-soft: rgba(102,212,146,0.12);

    --danger: #ff8585;

    --shadow:
        0 22px 55px rgba(0,0,0,0.30);

    --radius-lg: 24px;
    --radius-md: 15px;
    --radius-sm: 10px;
}


* {
    box-sizing: border-box;
}


html {
    min-height: 100%;
}


body {

    margin: 0;
    min-height: 100vh;

    font-family:
        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif;

    color: var(--text);

    background:

        radial-gradient(
            circle at 12% 8%,
            rgba(37,169,255,0.10),
            transparent 27%
        ),

        radial-gradient(
            circle at 88% 85%,
            rgba(102,212,146,0.06),
            transparent 28%
        ),

        var(--bg);
}


button,
input {
    font: inherit;
}


button {
    cursor: pointer;
}


.app-shell {

    width:
        min(
            1460px,
            calc(100% - 36px)
        );

    margin: 0 auto;

    padding:
        34px
        0
        60px;
}


/* ==========================================================
   HEADER
   ========================================================== */

.topbar {

    display: flex;

    justify-content:
        space-between;

    align-items:
        flex-start;

    gap: 30px;

    margin-bottom: 28px;
}


.eyebrow {

    margin:
        0
        0
        9px;

    color: var(--accent);

    font-size: 0.78rem;
    font-weight: 800;

    letter-spacing: 0.14em;

    text-transform: uppercase;
}


h1 {

    margin: 0;

    font-size:
        clamp(
            2.1rem,
            4vw,
            3.8rem
        );

    line-height: 0.98;

    letter-spacing: -0.055em;
}


.intro {

    max-width: 780px;

    margin:
        16px
        0
        0;

    color: var(--muted);

    font-size: 1rem;
    line-height: 1.65;
}


.dataset-badge {

    flex: 0 0 auto;

    padding:
        11px
        15px;

    color: var(--green);

    font-size: 0.84rem;
    font-weight: 750;

    background:
        var(--green-soft);

    border:
        1px solid
        rgba(102,212,146,0.25);

    border-radius: 999px;
}


/* ==========================================================
   MAIN LAYOUT
   ========================================================== */

.layout {

    display: grid;

    grid-template-columns:
        355px
        minmax(0, 1fr);

    gap: 24px;

    align-items: start;
}


.panel {

    background:

        linear-gradient(
            145deg,
            rgba(255,255,255,0.022),
            rgba(255,255,255,0)
        ),

        var(--panel);

    border:
        1px solid
        var(--border-soft);

    border-radius:
        var(--radius-lg);

    box-shadow:
        var(--shadow);
}


/* ==========================================================
   SIDEBAR
   ========================================================== */

.filters {

    position: sticky;
    top: 18px;

    max-height:
        calc(100vh - 36px);

    padding: 22px;

    overflow-y: auto;
}


.lookup-section {

    padding-bottom: 20px;

    border-bottom:
        1px solid
        var(--border-soft);

    margin-bottom: 20px;
}


.sidebar-heading {

    display: flex;

    align-items: center;

    justify-content:
        space-between;

    gap: 12px;

    margin-bottom: 14px;
}


.sidebar-heading h2 {

    margin: 0;

    font-size: 1.05rem;
}


.match-count {

    color: var(--accent);

    font-size: 0.80rem;

    font-weight: 750;
}


.filter-group {

    padding: 18px 0;

    border-top:
        1px solid
        var(--border-soft);
}


.filter-group.first {

    border-top: 0;
    padding-top: 0;
}


.filter-label {

    display: block;

    margin-bottom: 10px;

    color: var(--text);

    font-size: 0.85rem;

    font-weight: 720;
}


.filter-help {

    display: block;

    margin-top: 7px;

    color: var(--subtle);

    font-size: 0.72rem;

    line-height: 1.45;
}


input[type="number"],
input[type="text"] {

    width: 100%;

    min-width: 0;

    padding:
        11px
        12px;

    color: var(--text);

    background: #0c131e;

    border:
        1px solid
        var(--border);

    border-radius: 10px;

    outline: none;
}


input:focus {

    border-color:
        var(--accent-strong);

    box-shadow:
        0 0 0 3px
        rgba(37,169,255,0.12);
}


.input-pair {

    display: grid;

    grid-template-columns:
        1fr
        1fr;

    gap: 10px;
}


.input-caption {

    display: block;

    margin-bottom: 5px;

    color: var(--subtle);

    font-size: 0.70rem;
}


/* ==========================================================
   GAME LOOKUP
   ========================================================== */

.lookup-title {

    margin:
        0
        0
        10px;

    font-size: 0.92rem;
}


.lookup-wrap {

    position: relative;
}


.lookup-results {

    position: absolute;

    left: 0;
    right: 0;

    z-index: 50;

    max-height: 280px;

    margin-top: 6px;

    overflow-y: auto;

    background: #111a27;

    border:
        1px solid
        var(--border);

    border-radius: 12px;

    box-shadow:
        0 18px 35px
        rgba(0,0,0,0.35);
}


.lookup-result {

    display: block;

    width: 100%;

    padding:
        10px
        12px;

    text-align: left;

    color: var(--text);

    background: transparent;

    border: 0;

    border-bottom:
        1px solid
        var(--border-soft);
}


.lookup-result:last-child {
    border-bottom: 0;
}


.lookup-result:hover {
    background: var(--accent-soft);
}


.lookup-result-name {

    display: block;

    font-size: 0.82rem;

    font-weight: 720;
}


.lookup-result-meta {

    display: block;

    margin-top: 3px;

    color: var(--subtle);

    font-size: 0.70rem;
}


/* ==========================================================
   YEAR BUTTONS
   ========================================================== */

.year-grid {

    display: grid;

    grid-template-columns:
        repeat(
            2,
            minmax(0,1fr)
        );

    gap: 8px;
}


.year-option {
    position: relative;
}


.year-option input {

    position: absolute;

    opacity: 0;

    pointer-events: none;
}


.year-option span {

    display: flex;

    justify-content: center;
    align-items: center;

    min-height: 40px;

    color: var(--muted);

    font-weight: 720;

    background:
        var(--panel-2);

    border:
        1px solid
        var(--border);

    border-radius: 10px;
}


.year-option
input:checked
+
span {

    color: var(--accent);

    background:
        var(--accent-soft);

    border-color:
        rgba(37,169,255,0.50);
}


/* ==========================================================
   TAG BROWSER
   ========================================================== */

.tag-browser {
    margin-top: 12px;
}


.tag-browser summary {

    cursor: pointer;

    color: var(--accent);

    font-size: 0.77rem;

    font-weight: 750;
}


.available-tag-list {

    display: flex;

    flex-wrap: wrap;

    gap: 7px;

    max-height: 260px;

    margin-top: 10px;

    padding: 10px;

    overflow-y: auto;

    background: #0c131e;

    border:
        1px solid
        var(--border);

    border-radius: 12px;
}


.available-tag {

    padding:
        6px
        9px;

    color: var(--muted);

    font-size: 0.70rem;

    background:
        var(--panel-2);

    border:
        1px solid
        var(--border);

    border-radius: 999px;
}


.available-tag:hover,
.available-tag.selected {

    color: var(--accent);

    background:
        var(--accent-soft);

    border-color:
        rgba(37,169,255,0.55);
}


.selected-tags {

    display: flex;

    flex-wrap: wrap;

    gap: 6px;

    margin-top: 9px;
}


.selected-tag {

    display: inline-flex;

    align-items: center;

    gap: 5px;

    padding:
        5px
        8px;

    color: var(--accent);

    font-size: 0.70rem;

    font-weight: 700;

    background:
        var(--accent-soft);

    border:
        1px solid
        rgba(37,169,255,0.35);

    border-radius: 999px;
}


.selected-tag button {

    padding: 0;

    color: var(--accent);

    background: none;

    border: 0;
}


/* ==========================================================
   BUTTONS
   ========================================================== */

.button-stack {

    display: grid;

    gap: 9px;

    margin-top: 12px;
}


.primary-button,
.secondary-button,
.store-button,
.ghost-button {

    min-height: 44px;

    padding:
        0
        16px;

    font-weight: 780;

    border-radius: 11px;
}


.primary-button {

    color: #03111a;

    background:
        linear-gradient(
            135deg,
            #7ad0ff,
            #37abff
        );

    border: 0;
}


.primary-button:hover {
    filter: brightness(1.06);
}


.primary-button:disabled {

    cursor: not-allowed;

    opacity: 0.4;
}


.secondary-button {

    color: var(--text);

    background:
        var(--panel-2);

    border:
        1px solid
        var(--border);
}


.secondary-button:hover {

    background: #1a2637;

    border-color: #40536e;
}


.lookup-button {

    width: 100%;

    margin-top: 9px;
}


.ghost-button {

    min-height: 36px;

    color: var(--muted);

    font-size: 0.76rem;

    background: transparent;

    border:
        1px solid
        var(--border);
}


/* ==========================================================
   TOGGLE
   ========================================================== */

.toggle-row {

    display: flex;

    align-items: center;

    justify-content:
        space-between;

    gap: 14px;
}


.toggle-copy strong {

    display: block;

    font-size: 0.84rem;
}


.toggle-copy span {

    display: block;

    margin-top: 3px;

    color: var(--subtle);

    font-size: 0.71rem;
}


.switch {

    position: relative;

    width: 48px;
    height: 27px;

    flex: 0 0 auto;
}


.switch input {

    position: absolute;

    opacity: 0;
}


.switch-track {

    position: absolute;

    inset: 0;

    background: #29364b;

    border-radius: 999px;
}


.switch-track::after {

    content: "";

    position: absolute;

    top: 4px;
    left: 4px;

    width: 19px;
    height: 19px;

    background: white;

    border-radius: 50%;

    transition:
        transform 160ms ease;
}


.switch
input:checked
+
.switch-track {

    background:
        var(--accent-strong);
}


.switch
input:checked
+
.switch-track::after {

    transform:
        translateX(21px);
}


/* ==========================================================
   RESULT
   ========================================================== */

.result-column {
    min-width: 0;
}


.empty-state {

    min-height: 650px;

    display: grid;

    place-items: center;

    padding: 50px;

    text-align: center;
}


.empty-icon {

    width: 76px;
    height: 76px;

    margin:
        0 auto 20px;

    display: grid;

    place-items: center;

    color: var(--accent);

    font-size: 2rem;

    background:
        var(--accent-soft);

    border:
        1px solid
        rgba(37,169,255,0.25);

    border-radius: 22px;
}


.empty-state h2 {

    margin: 0;

    font-size: 1.55rem;
}


.empty-state p {

    max-width: 560px;

    margin:
        12px auto 0;

    color: var(--muted);

    line-height: 1.65;
}


.hidden {
    display: none !important;
}


/* ==========================================================
   CASE STUDY HERO
   ========================================================== */

.case-study {
    overflow: hidden;
}


.hero {

    position: relative;

    min-height: 440px;

    overflow: hidden;

    background: #090e17;
}


.hero-image {

    display: block;

    width: 100%;
    height: 440px;

    object-fit: cover;
}


.hero-image.loading {

    opacity: 0.35;

    filter: blur(5px);
}


.hero-gradient {

    position: absolute;

    inset: 0;

    background:

        linear-gradient(
            to top,
            rgba(7,10,17,0.97),
            rgba(7,10,17,0.42) 48%,
            rgba(7,10,17,0.05) 78%
        );
}


.hero-content {

    position: absolute;

    left: 30px;
    right: 30px;
    bottom: 27px;
}


.case-label {

    display: inline-flex;

    margin-bottom: 12px;

    padding:
        7px
        10px;

    color: var(--green);

    font-size: 0.71rem;

    font-weight: 800;

    letter-spacing: 0.09em;

    text-transform: uppercase;

    background:
        rgba(8,12,19,0.74);

    border:
        1px solid
        rgba(102,212,146,0.32);

    border-radius: 999px;
}


.game-title {

    max-width: 1000px;

    margin: 0;

    font-size:
        clamp(
            2rem,
            5vw,
            4.3rem
        );

    line-height: 0.98;

    letter-spacing: -0.055em;
}


.hero-meta {

    display: flex;

    flex-wrap: wrap;

    gap: 8px;

    margin-top: 15px;
}


.hero-chip {

    padding:
        7px
        10px;

    color: #dce7f5;

    font-size: 0.76rem;

    font-weight: 700;

    background:
        rgba(8,12,19,0.75);

    border:
        1px solid
        rgba(255,255,255,0.13);

    border-radius: 999px;
}


/* ==========================================================
   CASE STUDY CONTENT
   ========================================================== */

.case-content {
    padding: 27px;
}


.metric-grid {

    display: grid;

    grid-template-columns:
        repeat(
            3,
            minmax(0,1fr)
        );

    gap: 11px;
}


.metric {

    min-width: 0;

    padding: 15px;

    background:
        var(--panel-2);

    border:
        1px solid
        var(--border-soft);

    border-radius: 14px;
}


.metric-label {

    display: block;

    margin-bottom: 7px;

    color: var(--subtle);

    font-size: 0.69rem;

    font-weight: 700;

    text-transform: uppercase;

    letter-spacing: 0.07em;
}


.metric-value {

    display: block;

    color: var(--text);

    font-size:
        clamp(
            1rem,
            2vw,
            1.35rem
        );

    font-weight: 820;
}


.metric-value.accent {
    color: var(--accent);
}


.metric-value.green {
    color: var(--green);
}


.details-grid {

    display: grid;

    grid-template-columns:
        minmax(0,1.55fr)
        minmax(250px,0.72fr);

    gap: 26px;

    margin-top: 27px;
}


.section-title {

    margin:
        0
        0
        11px;

    font-size: 0.88rem;
}


.description {

    margin: 0;

    color: #cad5e5;

    font-size: 1rem;

    line-height: 1.72;
}


.tag-list {

    display: flex;

    flex-wrap: wrap;

    gap: 7px;
}


.tag {

    padding:
        7px
        10px;

    color: #bdc9da;

    font-size: 0.74rem;

    font-weight: 650;

    background: #0d141f;

    border:
        1px solid
        var(--border);

    border-radius: 999px;
}


.rank-card {

    padding: 18px;

    background:

        linear-gradient(
            145deg,
            rgba(37,169,255,0.10),
            rgba(37,169,255,0.02)
        );

    border:
        1px solid
        rgba(37,169,255,0.24);

    border-radius: 16px;
}


.rank-label {

    color: var(--muted);

    font-size: 0.75rem;
}


.rank-value {

    display: block;

    margin-top: 7px;

    color: var(--accent);

    font-size: 2rem;

    font-weight: 860;

    letter-spacing: -0.04em;
}


.rank-note {

    display: block;

    margin-top: 6px;

    color: var(--subtle);

    font-size: 0.73rem;

    line-height: 1.45;
}


.actions {

    display: flex;

    flex-wrap: wrap;

    gap: 10px;

    margin-top: 27px;

    padding-top: 23px;

    border-top:
        1px solid
        var(--border-soft);
}


.store-button {

    display: inline-flex;

    align-items: center;

    justify-content: center;

    color: #04110b;

    text-decoration: none;

    background: var(--green);

    border: 0;
}


/* ==========================================================
   HISTORY
   ========================================================== */

.history-panel {

    margin-top: 18px;

    padding: 19px;
}


.history-list {

    display: grid;

    gap: 8px;
}


.history-item {

    display: grid;

    grid-template-columns:
        36px
        minmax(0,1fr)
        auto;

    gap: 11px;

    align-items: center;

    padding:
        9px
        11px;

    background:
        var(--panel-2);

    border:
        1px solid
        var(--border-soft);

    border-radius: 11px;
}


.history-number {

    color: var(--accent);

    font-size: 0.75rem;

    font-weight: 800;
}


.history-name {

    overflow: hidden;

    text-overflow: ellipsis;

    white-space: nowrap;

    font-size: 0.82rem;

    font-weight: 700;
}


.history-revenue {

    color: var(--muted);

    font-size: 0.73rem;
}


/* ==========================================================
   TOAST
   ========================================================== */

.toast {

    position: fixed;

    left: 50%;
    bottom: 26px;

    z-index: 100;

    padding:
        10px
        15px;

    color: var(--text);

    font-size: 0.82rem;

    font-weight: 700;

    background: #172234;

    border:
        1px solid
        #364962;

    border-radius: 999px;

    box-shadow: var(--shadow);

    opacity: 0;

    pointer-events: none;

    transform:
        translate(
            -50%,
            12px
        );

    transition:
        160ms ease;
}


.toast.visible {

    opacity: 1;

    transform:
        translate(
            -50%,
            0
        );
}


/* ==========================================================
   RESPONSIVE
   ========================================================== */

@media (max-width: 1050px) {

    .layout {
        grid-template-columns: 1fr;
    }

    .filters {

        position: static;

        max-height: none;
    }

}


@media (max-width: 700px) {

    .app-shell {

        width:
            calc(100% - 20px);

        padding-top: 20px;
    }

    .topbar {
        display: block;
    }

    .dataset-badge {

        display: inline-flex;

        margin-top: 18px;
    }

    .metric-grid {

        grid-template-columns:
            repeat(
                2,
                minmax(0,1fr)
            );
    }

    .details-grid {
        grid-template-columns: 1fr;
    }

    .input-pair {
        grid-template-columns: 1fr;
    }

    .hero,
    .hero-image {
        height: 360px;
        min-height: 360px;
    }

}

</style>

</head>


<body>


<main class="app-shell">


<header class="topbar">

    <div>

        <p class="eyebrow">
            Steam market study tool
        </p>

        <h1>
            Indie Case Study Suggester
        </h1>

        <p class="intro">
            Filter the recent Steam market and draw a random game
            to study its positioning, pricing, commercial outcome,
            tags, artwork, review score, and store pitch.
        </p>

    </div>


    <div class="dataset-badge">

        __GAME_COUNT__
        qualifying games

    </div>

</header>



<div class="layout">


<!-- ========================================================
     SIDEBAR
     ======================================================== -->

<aside class="panel filters">


    <!-- DIRECT LOOKUP -->

    <section class="lookup-section">

        <h2 class="lookup-title">
            Find a specific game
        </h2>

        <div class="lookup-wrap">

            <input
                id="gameLookup"
                type="text"
                placeholder="Game name or AppID"
                autocomplete="off"
            >

            <div
                id="lookupResults"
                class="lookup-results hidden"
            ></div>

        </div>


        <button
            id="loadGameButton"
            class="secondary-button lookup-button"
            type="button"
        >
            Load game report
        </button>


        <span class="filter-help">

            This ignores the study filters and loads
            the exact game's report.

        </span>

    </section>



    <!-- FILTER HEADING -->

    <div class="sidebar-heading">

        <h2>
            Study filters
        </h2>

        <span
            id="matchCount"
            class="match-count"
        >
            Calculating…
        </span>

    </div>



    <!-- RELEASE YEAR -->

    <section class="filter-group first">

        <label class="filter-label">

            Release years

        </label>


        <div class="year-grid">

            __YEAR_CHECKBOXES__

        </div>

    </section>



    <!-- REVENUE -->

    <section class="filter-group">

        <label class="filter-label">

            Estimated revenue

        </label>


        <div class="input-pair">

            <label>

                <span class="input-caption">
                    Minimum
                </span>

                <input
                    id="minRevenue"
                    type="number"
                    min="0"
                    step="10000"
                    value="150000"
                >

            </label>


            <label>

                <span class="input-caption">
                    Maximum
                </span>

                <input
                    id="maxRevenue"
                    type="number"
                    min="0"
                    step="10000"
                    value="800000"
                >

            </label>

        </div>


        <span class="filter-help">

            Estimated as total reviews × 45 × listed price.

        </span>

    </section>



    <!-- PRICE -->

    <section class="filter-group">

        <label class="filter-label">

            Listed price

        </label>


        <div class="input-pair">

            <label>

                <span class="input-caption">
                    Minimum
                </span>

                <input
                    id="minPrice"
                    type="number"
                    min="0"
                    value="0"
                >

            </label>


            <label>

                <span class="input-caption">
                    Maximum
                </span>

                <input
                    id="maxPrice"
                    type="number"
                    min="0"
                    value="__MAX_PRICE__"
                >

            </label>

        </div>

    </section>



    <!-- RANK -->

    <section class="filter-group">

        <label class="filter-label">

            Revenue rank in release year

        </label>


        <div class="input-pair">

            <label>

                <span class="input-caption">
                    Best rank
                </span>

                <input
                    id="minRank"
                    type="number"
                    min="1"
                    value="1"
                >

            </label>


            <label>

                <span class="input-caption">
                    Worst rank
                </span>

                <input
                    id="maxRank"
                    type="number"
                    min="1"
                    value="__MAX_RANK__"
                >

            </label>

        </div>

    </section>



    <!-- PERCENTILE -->

    <section class="filter-group">

        <label class="filter-label">

            Revenue percentile

        </label>


        <div class="input-pair">

            <label>

                <span class="input-caption">
                    Minimum
                </span>

                <input
                    id="minPercentile"
                    type="number"
                    min="0"
                    max="100"
                    value="0"
                >

            </label>


            <label>

                <span class="input-caption">
                    Maximum
                </span>

                <input
                    id="maxPercentile"
                    type="number"
                    min="0"
                    max="100"
                    value="100"
                >

            </label>

        </div>

    </section>



    <!-- REVIEW COUNT -->

    <section class="filter-group">

        <label class="filter-label">

            Review count

        </label>


        <div class="input-pair">

            <label>

                <span class="input-caption">
                    Minimum
                </span>

                <input
                    id="minReviews"
                    type="number"
                    min="20"
                    value="20"
                >

            </label>


            <label>

                <span class="input-caption">
                    Maximum
                </span>

                <input
                    id="maxReviews"
                    type="number"
                    min="20"
                    placeholder="No limit"
                >

            </label>

        </div>

    </section>



    <!-- REVIEW SCORE -->

    <section class="filter-group">

        <label class="filter-label">

            Positive review score

        </label>


        <div class="input-pair">

            <label>

                <span class="input-caption">
                    Minimum %
                </span>

                <input
                    id="minReviewScore"
                    type="number"
                    min="0"
                    max="100"
                    value="50"
                >

            </label>


            <label>

                <span class="input-caption">
                    Maximum %
                </span>

                <input
                    id="maxReviewScore"
                    type="number"
                    min="0"
                    max="100"
                    value="100"
                >

            </label>

        </div>

    </section>



    <!-- INCLUDED TAGS -->

    <section class="filter-group">

        <label class="filter-label">

            Must include tags

        </label>


        <input
            id="includeTags"
            type="text"
            placeholder="e.g. Colony Sim, Roguelike"
        >


        <div
            id="selectedTags"
            class="selected-tags"
        ></div>


        <details class="tag-browser">

            <summary>

                Browse all __TAG_COUNT__ tags

            </summary>


            <input
                id="tagSearch"
                type="text"
                placeholder="Search available tags"
                style="margin-top:10px;"
            >


            <div
                id="availableTagList"
                class="available-tag-list"
            >

                __TAG_BUTTONS__

            </div>

        </details>

    </section>



    <!-- EXCLUDED TAGS -->

    <section class="filter-group">

        <label class="filter-label">

            Exclude tags

        </label>


        <input
            id="excludeTags"
            type="text"
            placeholder="e.g. VR, Multiplayer"
        >

    </section>



    <!-- AVOID REPEATS -->

    <section class="filter-group">

        <div class="toggle-row">

            <div class="toggle-copy">

                <strong>
                    Avoid repeats
                </strong>

                <span>
                    Skip games already shown this session.
                </span>

            </div>


            <label class="switch">

                <input
                    id="avoidRepeats"
                    type="checkbox"
                    checked
                >

                <span class="switch-track"></span>

            </label>

        </div>

    </section>



    <!-- BUTTONS -->

    <div class="button-stack">

        <button
            id="suggestButton"
            class="primary-button"
            type="button"
        >
            Suggest a case study
        </button>


        <button
            id="resetButton"
            class="secondary-button"
            type="button"
        >
            Reset filters
        </button>

    </div>


</aside>



<!-- ========================================================
     RESULT AREA
     ======================================================== -->

<section class="result-column">


    <!-- EMPTY STATE -->

    <div
        id="emptyState"
        class="panel empty-state"
    >

        <div>

            <div class="empty-icon">
                ◈
            </div>


            <h2>

                Your next case study is waiting

            </h2>


            <p>

                The default segment contains games earning an
                estimated $150K–$800K. Narrow the market by year,
                review score, price, rank, percentile, or Steam tags.

            </p>

        </div>

    </div>



    <!-- CASE STUDY -->

    <article
        id="caseStudy"
        class="panel case-study hidden"
    >


        <div class="hero">

            <img
                id="heroImage"
                class="hero-image"
                alt=""
            >


            <div class="hero-gradient"></div>


            <div class="hero-content">

                <div class="case-label">

                    Indie case study

                </div>


                <h2
                    id="gameTitle"
                    class="game-title"
                ></h2>


                <div
                    id="heroMeta"
                    class="hero-meta"
                ></div>

            </div>

        </div>



        <div class="case-content">


            <!-- METRICS -->

            <div class="metric-grid">


                <div class="metric">

                    <span class="metric-label">
                        Estimated revenue
                    </span>

                    <span
                        id="gameRevenue"
                        class="metric-value green"
                    ></span>

                </div>


                <div class="metric">

                    <span class="metric-label">
                        Listed price
                    </span>

                    <span
                        id="gamePrice"
                        class="metric-value"
                    ></span>

                </div>


                <div class="metric">

                    <span class="metric-label">
                        Yearly revenue rank
                    </span>

                    <span
                        id="gameRank"
                        class="metric-value accent"
                    ></span>

                </div>


                <div class="metric">

                    <span class="metric-label">
                        Revenue percentile
                    </span>

                    <span
                        id="gamePercentile"
                        class="metric-value"
                    ></span>

                </div>


                <div class="metric">

                    <span class="metric-label">
                        Steam reviews
                    </span>

                    <span
                        id="gameReviews"
                        class="metric-value"
                    ></span>

                </div>


                <div class="metric">

                    <span class="metric-label">
                        Positive reviews
                    </span>

                    <span
                        id="gameReviewScore"
                        class="metric-value"
                    ></span>

                </div>


            </div>



            <!-- DETAILS -->

            <div class="details-grid">


                <div>


                    <section>

                        <h3 class="section-title">

                            Store pitch

                        </h3>


                        <p
                            id="gameDescription"
                            class="description"
                        ></p>

                    </section>



                    <section style="margin-top:26px;">

                        <h3 class="section-title">

                            Steam tags

                        </h3>


                        <div
                            id="gameTags"
                            class="tag-list"
                        ></div>

                    </section>


                </div>



                <aside>


                    <div class="rank-card">

                        <span class="rank-label">

                            Position among qualifying releases

                        </span>


                        <strong
                            id="rankCardValue"
                            class="rank-value"
                        ></strong>


                        <span
                            id="rankCardNote"
                            class="rank-note"
                        ></span>

                    </div>


                </aside>


            </div>



            <!-- ACTIONS -->

            <div class="actions">


                <button
                    id="anotherButton"
                    class="primary-button"
                    type="button"
                >

                    Another case study

                </button>


                <a
                    id="steamLink"
                    class="store-button"
                    target="_blank"
                    rel="noopener noreferrer"
                >

                    Reveal Steam page ↗

                </a>


                <button
                    id="copyAppIdButton"
                    class="secondary-button"
                    type="button"
                >

                    Copy AppID

                </button>


            </div>


        </div>


    </article>



    <!-- HISTORY -->

    <section
        id="historyPanel"
        class="panel history-panel hidden"
    >


        <div class="sidebar-heading">


            <h2>
                Session history
            </h2>


            <button
                id="clearHistoryButton"
                class="ghost-button"
                type="button"
            >

                Clear

            </button>


        </div>


        <div
            id="historyList"
            class="history-list"
        ></div>


    </section>


</section>


</div>


</main>



<div
    id="toast"
    class="toast"
    role="status"
    aria-live="polite"
></div>



<script
    id="game-data"
    type="application/json"
>
__GAME_DATA__
</script>



<script>

"use strict";


/* ==========================================================
   DATA
   ========================================================== */

const games = JSON.parse(
    document
        .getElementById("game-data")
        .textContent
);



/* ==========================================================
   ELEMENT REFERENCES
   ========================================================== */

const elements = {

    yearCheckboxes: [
        ...document.querySelectorAll(
            ".year-checkbox"
        )
    ],

    gameLookup:
        document.getElementById(
            "gameLookup"
        ),

    lookupResults:
        document.getElementById(
            "lookupResults"
        ),

    loadGameButton:
        document.getElementById(
            "loadGameButton"
        ),

    minRevenue:
        document.getElementById(
            "minRevenue"
        ),

    maxRevenue:
        document.getElementById(
            "maxRevenue"
        ),

    minPrice:
        document.getElementById(
            "minPrice"
        ),

    maxPrice:
        document.getElementById(
            "maxPrice"
        ),

    minRank:
        document.getElementById(
            "minRank"
        ),

    maxRank:
        document.getElementById(
            "maxRank"
        ),

    minPercentile:
        document.getElementById(
            "minPercentile"
        ),

    maxPercentile:
        document.getElementById(
            "maxPercentile"
        ),

    minReviews:
        document.getElementById(
            "minReviews"
        ),

    maxReviews:
        document.getElementById(
            "maxReviews"
        ),

    minReviewScore:
        document.getElementById(
            "minReviewScore"
        ),

    maxReviewScore:
        document.getElementById(
            "maxReviewScore"
        ),

    includeTags:
        document.getElementById(
            "includeTags"
        ),

    excludeTags:
        document.getElementById(
            "excludeTags"
        ),

    selectedTags:
        document.getElementById(
            "selectedTags"
        ),

    tagSearch:
        document.getElementById(
            "tagSearch"
        ),

    availableTagButtons: [
        ...document.querySelectorAll(
            ".available-tag"
        )
    ],

    avoidRepeats:
        document.getElementById(
            "avoidRepeats"
        ),

    matchCount:
        document.getElementById(
            "matchCount"
        ),

    suggestButton:
        document.getElementById(
            "suggestButton"
        ),

    resetButton:
        document.getElementById(
            "resetButton"
        ),

    emptyState:
        document.getElementById(
            "emptyState"
        ),

    caseStudy:
        document.getElementById(
            "caseStudy"
        ),

    heroImage:
        document.getElementById(
            "heroImage"
        ),

    gameTitle:
        document.getElementById(
            "gameTitle"
        ),

    heroMeta:
        document.getElementById(
            "heroMeta"
        ),

    gameRevenue:
        document.getElementById(
            "gameRevenue"
        ),

    gamePrice:
        document.getElementById(
            "gamePrice"
        ),

    gameRank:
        document.getElementById(
            "gameRank"
        ),

    gamePercentile:
        document.getElementById(
            "gamePercentile"
        ),

    gameReviews:
        document.getElementById(
            "gameReviews"
        ),

    gameReviewScore:
        document.getElementById(
            "gameReviewScore"
        ),

    gameDescription:
        document.getElementById(
            "gameDescription"
        ),

    gameTags:
        document.getElementById(
            "gameTags"
        ),

    rankCardValue:
        document.getElementById(
            "rankCardValue"
        ),

    rankCardNote:
        document.getElementById(
            "rankCardNote"
        ),

    anotherButton:
        document.getElementById(
            "anotherButton"
        ),

    steamLink:
        document.getElementById(
            "steamLink"
        ),

    copyAppIdButton:
        document.getElementById(
            "copyAppIdButton"
        ),

    historyPanel:
        document.getElementById(
            "historyPanel"
        ),

    historyList:
        document.getElementById(
            "historyList"
        ),

    clearHistoryButton:
        document.getElementById(
            "clearHistoryButton"
        ),

    toast:
        document.getElementById(
            "toast"
        )

};



/* ==========================================================
   STATE
   ========================================================== */

const state = {

    currentGame: null,

    shownAppIds:
        new Set(),

    history: []

};



/* ==========================================================
   HELPERS
   ========================================================== */

function numericValue(
    element,
    fallback
) {

    if (
        element.value === ""
        ||
        element.value === null
    ) {
        return fallback;
    }

    const value =
        Number(
            element.value
        );

    return Number.isFinite(value)
        ? value
        : fallback;
}



function escapeHtml(value) {

    const container =
        document.createElement("div");

    container.textContent =
        String(value ?? "");

    return container.innerHTML;
}



function selectedYears() {

    return new Set(

        elements
            .yearCheckboxes

            .filter(
                checkbox =>
                    checkbox.checked
            )

            .map(
                checkbox =>
                    Number(
                        checkbox.value
                    )
            )

    );

}



function normalizedTags(game) {

    if (
        Array.isArray(
            game.tags
        )
    ) {

        return game.tags
            .map(
                tag =>
                    String(tag).trim()
            )
            .filter(Boolean);

    }


    if (
        game.tags
        &&
        typeof game.tags
            === "object"
    ) {

        return Object.keys(
            game.tags
        );

    }


    return [];

}



function parseTags(value) {

    return String(value || "")

        .split(",")

        .map(
            tag =>
                tag
                    .trim()
                    .toLocaleLowerCase()
        )

        .filter(Boolean);

}



function displayTags(value) {

    return String(value || "")

        .split(",")

        .map(
            tag =>
                tag.trim()
        )

        .filter(Boolean);

}



function formatCurrency(value) {

    return new Intl.NumberFormat(

        "en-US",

        {

            style:
                "currency",

            currency:
                "USD",

            maximumFractionDigits:
                Number(value) >= 1000
                    ? 0
                    : 2

        }

    ).format(
        Number(value || 0)
    );

}



function formatCompactCurrency(value) {

    const number =
        Number(value || 0);


    if (
        number >= 1_000_000_000
    ) {

        return (
            "$"
            +
            (
                number
                /
                1_000_000_000
            ).toFixed(1)
            +
            "B"
        );

    }


    if (
        number >= 1_000_000
    ) {

        return (
            "$"
            +
            (
                number
                /
                1_000_000
            ).toFixed(1)
            +
            "M"
        );

    }


    if (
        number >= 1_000
    ) {

        return (
            "$"
            +
            Math.round(
                number / 1_000
            )
            +
            "K"
        );

    }


    return formatCurrency(
        number
    );

}



/* ==========================================================
   DIRECT GAME LOOKUP
   ========================================================== */

function searchGames(query) {

    const cleaned =
        String(query || "")
            .trim()
            .toLocaleLowerCase();


    if (!cleaned) {
        return [];
    }


    /* Exact AppID gets priority */

    if (/^\d+$/.test(cleaned)) {

        const appid =
            Number(cleaned);

        const exactApp =
            games.find(
                game =>
                    Number(
                        game.appid
                    )
                    === appid
            );

        return exactApp
            ? [exactApp]
            : [];

    }


    const matches =
        games.filter(
            game =>

                String(
                    game.name || ""
                )
                .toLocaleLowerCase()
                .includes(cleaned)
        );


    matches.sort(
        (a, b) => {

            const aName =
                String(
                    a.name || ""
                ).toLocaleLowerCase();

            const bName =
                String(
                    b.name || ""
                ).toLocaleLowerCase();


            const aStarts =
                aName.startsWith(
                    cleaned
                );

            const bStarts =
                bName.startsWith(
                    cleaned
                );


            if (
                aStarts
                !==
                bStarts
            ) {

                return aStarts
                    ? -1
                    : 1;

            }


            return (
                Number(
                    a.revenue_rank_in_year
                    || Infinity
                )
                -
                Number(
                    b.revenue_rank_in_year
                    || Infinity
                )
            );

        }
    );


    return matches.slice(
        0,
        12
    );

}



function renderLookupResults() {

    const query =
        elements
            .gameLookup
            .value;


    const matches =
        searchGames(
            query
        );


    if (
        matches.length === 0
    ) {

        elements
            .lookupResults
            .classList
            .add("hidden");

        elements
            .lookupResults
            .innerHTML = "";

        return;

    }


    elements
        .lookupResults
        .innerHTML =

        matches.map(
            game => `

                <button
                    class="lookup-result"
                    type="button"
                    data-appid="${game.appid}"
                >

                    <span
                        class="lookup-result-name"
                    >
                        ${escapeHtml(
                            game.name
                        )}
                    </span>

                    <span
                        class="lookup-result-meta"
                    >
                        ${game.release_year}
                        · AppID ${game.appid}
                        · ${formatCompactCurrency(
                            game.estimated_revenue
                        )}
                    </span>

                </button>

            `
        ).join("");


    elements
        .lookupResults
        .classList
        .remove("hidden");


    elements
        .lookupResults
        .querySelectorAll(
            ".lookup-result"
        )
        .forEach(
            button => {

                button.addEventListener(
                    "click",
                    () => {

                        const appid =
                            Number(
                                button.dataset.appid
                            );

                        const game =
                            games.find(
                                candidate =>
                                    Number(
                                        candidate.appid
                                    )
                                    === appid
                            );


                        if (game) {

                            elements
                                .gameLookup
                                .value =
                                game.name;

                            elements
                                .lookupResults
                                .classList
                                .add("hidden");

                            renderGame(
                                game
                            );

                        }

                    }
                );

            }
        );

}



function findExactGame(query) {

    const cleaned =
        String(query || "")
            .trim();


    if (!cleaned) {
        return null;
    }


    if (/^\d+$/.test(cleaned)) {

        const appid =
            Number(cleaned);

        const appMatch =
            games.find(
                game =>
                    Number(
                        game.appid
                    )
                    === appid
            );


        if (appMatch) {
            return appMatch;
        }

    }


    const normalized =
        cleaned
            .toLocaleLowerCase();


    const exactName =
        games.find(
            game =>
                String(
                    game.name || ""
                )
                .trim()
                .toLocaleLowerCase()
                === normalized
        );


    if (exactName) {
        return exactName;
    }


    const matches =
        searchGames(
            cleaned
        );


    if (
        matches.length > 0
    ) {

        return matches[0];

    }


    return null;

}



function loadSpecificGame() {

    const game =
        findExactGame(
            elements
                .gameLookup
                .value
        );


    if (!game) {

        showToast(
            "No matching game found."
        );

        return;

    }


    elements
        .gameLookup
        .value =
        game.name;


    elements
        .lookupResults
        .classList
        .add("hidden");


    renderGame(
        game
    );


    showToast(
        `Loaded ${game.name}.`
    );

}



/* ==========================================================
   TAG FILTER UI
   ========================================================== */

function addIncludedTag(tag) {

    const selected =
        displayTags(
            elements
                .includeTags
                .value
        );


    const normalized =
        selected.map(
            value =>
                value
                    .toLocaleLowerCase()
        );


    if (
        !normalized.includes(
            tag.toLocaleLowerCase()
        )
    ) {

        selected.push(
            tag
        );

    }


    elements
        .includeTags
        .value =
        selected.join(", ");


    updateTagUI();

    updateMatchCount();

}



function removeIncludedTag(tag) {

    const remaining =
        displayTags(
            elements
                .includeTags
                .value
        )

        .filter(
            value =>
                value
                    .toLocaleLowerCase()
                !==
                tag
                    .toLocaleLowerCase()
        );


    elements
        .includeTags
        .value =
        remaining.join(", ");


    updateTagUI();

    updateMatchCount();

}



function updateTagUI() {

    const chosen =
        displayTags(
            elements
                .includeTags
                .value
        );


    const normalized =
        chosen.map(
            tag =>
                tag
                    .toLocaleLowerCase()
        );


    elements
        .availableTagButtons
        .forEach(
            button => {

                button
                    .classList
                    .toggle(

                        "selected",

                        normalized.includes(
                            button
                                .dataset
                                .tag
                                .toLocaleLowerCase()
                        )

                    );

            }
        );


    elements
        .selectedTags
        .innerHTML =

        chosen.map(
            tag => `

                <span class="selected-tag">

                    ${escapeHtml(tag)}

                    <button
                        type="button"
                        data-remove-tag="${escapeHtml(tag)}"
                    >
                        ×
                    </button>

                </span>

            `
        ).join("");


    elements
        .selectedTags
        .querySelectorAll(
            "[data-remove-tag]"
        )
        .forEach(
            button => {

                button.addEventListener(
                    "click",
                    () => {

                        removeIncludedTag(
                            button
                                .dataset
                                .removeTag
                        );

                    }
                );

            }
        );

}



function filterTagBrowser() {

    const query =
        elements
            .tagSearch
            .value
            .trim()
            .toLocaleLowerCase();


    elements
        .availableTagButtons
        .forEach(
            button => {

                const tag =
                    button
                        .dataset
                        .tag
                        .toLocaleLowerCase();


                button.hidden =
                    query
                    &&
                    !tag.includes(
                        query
                    );

            }
        );

}



/* ==========================================================
   FILTERING
   ========================================================== */

function filteredGames(
    {
        respectHistory = true
    } = {}
) {

    const years =
        selectedYears();


    const minRevenue =
        numericValue(
            elements.minRevenue,
            0
        );


    const maxRevenue =
        numericValue(
            elements.maxRevenue,
            Infinity
        );


    const minPrice =
        numericValue(
            elements.minPrice,
            0
        );


    const maxPrice =
        numericValue(
            elements.maxPrice,
            Infinity
        );


    const minRank =
        numericValue(
            elements.minRank,
            1
        );


    const maxRank =
        numericValue(
            elements.maxRank,
            Infinity
        );


    const minPercentile =
        numericValue(
            elements.minPercentile,
            0
        );


    const maxPercentile =
        numericValue(
            elements.maxPercentile,
            100
        );


    const minReviews =
        numericValue(
            elements.minReviews,
            20
        );


    const maxReviews =
        numericValue(
            elements.maxReviews,
            Infinity
        );


    const minReviewScore =
        numericValue(
            elements.minReviewScore,
            0
        );


    const maxReviewScore =
        numericValue(
            elements.maxReviewScore,
            100
        );


    const requiredTags =
        parseTags(
            elements
                .includeTags
                .value
        );


    const excludedTags =
        parseTags(
            elements
                .excludeTags
                .value
        );


    return games.filter(
        game => {

            const appid =
                Number(
                    game.appid
                );


            const gameYear =
                Number(
                    game.release_year
                );


            const revenue =
                Number(
                    game.estimated_revenue
                    || 0
                );


            const price =
                Number(
                    game.price
                    || 0
                );


            const rank =
                Number(
                    game.revenue_rank_in_year
                    || 0
                );


            const percentile =
                Number(
                    game.revenue_percentile_in_year
                    || 0
                );


            const reviews =
                Number(
                    game.total_reviews
                    || 0
                );


            const reviewScore =
                Number(
                    game.review_score
                    || 0
                );


            const tags =
                normalizedTags(game)
                    .map(
                        tag =>
                            tag
                            .toLocaleLowerCase()
                    );


            const hasRequired =
                requiredTags.every(
                    required =>
                        tags.some(
                            tag =>
                                tag === required
                                ||
                                tag.includes(
                                    required
                                )
                        )
                );


            const hasExcluded =
                excludedTags.some(
                    excluded =>
                        tags.some(
                            tag =>
                                tag === excluded
                                ||
                                tag.includes(
                                    excluded
                                )
                        )
                );


            const repeated =
                respectHistory
                &&
                elements
                    .avoidRepeats
                    .checked
                &&
                state
                    .shownAppIds
                    .has(appid);


            return (

                years.has(
                    gameYear
                )

                &&

                revenue >= minRevenue

                &&

                revenue <= maxRevenue

                &&

                price >= minPrice

                &&

                price <= maxPrice

                &&

                rank >= minRank

                &&

                rank <= maxRank

                &&

                percentile >=
                    minPercentile

                &&

                percentile <=
                    maxPercentile

                &&

                reviews >=
                    minReviews

                &&

                reviews <=
                    maxReviews

                &&

                reviewScore >=
                    minReviewScore

                &&

                reviewScore <=
                    maxReviewScore

                &&

                hasRequired

                &&

                !hasExcluded

                &&

                !repeated

            );

        }
    );

}



function updateMatchCount() {

    const matches =
        filteredGames();


    if (
        matches.length === 0
        &&
        elements
            .avoidRepeats
            .checked
    ) {

        const withoutHistory =
            filteredGames({
                respectHistory: false
            });


        if (
            withoutHistory.length > 0
        ) {

            elements
                .matchCount
                .textContent =
                "All seen";

            elements
                .suggestButton
                .disabled =
                false;

            return;

        }

    }


    elements
        .matchCount
        .textContent =

        `${matches.length.toLocaleString()} matches`;


    elements
        .suggestButton
        .disabled =
        matches.length === 0;

}



/* ==========================================================
   STEAM ARTWORK
   ========================================================== */

function marketingImageUrls(appid) {

    return [

        `https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/${appid}/header.jpg`,

        `https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/${appid}/capsule_616x353.jpg`,

        `https://cdn.akamai.steamstatic.com/steam/apps/${appid}/header.jpg`

    ];

}



function loadMarketingImage(game) {

    const urls =
        marketingImageUrls(
            game.appid
        );


    let index = 0;


    elements
        .heroImage
        .classList
        .add("loading");


    elements
        .heroImage
        .alt =
        `${game.name} Steam artwork`;


    function tryNext() {

        if (
            index >=
            urls.length
        ) {

            elements
                .heroImage
                .onerror =
                null;


            elements
                .heroImage
                .src =

                "data:image/svg+xml;charset=UTF-8,"
                +
                encodeURIComponent(`

                    <svg
                        xmlns="http://www.w3.org/2000/svg"
                        width="1200"
                        height="600"
                    >

                        <rect
                            width="1200"
                            height="600"
                            fill="#0d1420"
                        />

                        <text
                            x="600"
                            y="300"
                            fill="#7f91aa"
                            font-family="Arial"
                            font-size="34"
                            text-anchor="middle"
                        >
                            Steam artwork unavailable
                        </text>

                    </svg>

                `);


            elements
                .heroImage
                .classList
                .remove("loading");


            return;

        }


        elements
            .heroImage
            .src =
            urls[index];


        index += 1;

    }


    elements
        .heroImage
        .onload =
        () => {

            elements
                .heroImage
                .classList
                .remove("loading");

        };


    elements
        .heroImage
        .onerror =
        tryNext;


    tryNext();

}



/* ==========================================================
   RENDER CASE STUDY
   ========================================================== */

function renderGame(game) {

    state.currentGame =
        game;


    const tags =
        normalizedTags(game);


    elements
        .gameTitle
        .textContent =
        game.name;


    const chips = [

        `<span class="hero-chip">
            ${escapeHtml(
                game.release_year
            )}
        </span>`,

        `<span class="hero-chip">
            AppID
            ${escapeHtml(
                game.appid
            )}
        </span>`

    ];


    if (
        tags.length > 0
    ) {

        chips.push(

            `<span class="hero-chip">
                ${escapeHtml(
                    tags
                    .slice(0, 3)
                    .join(" · ")
                )}
            </span>`

        );

    }


    elements
        .heroMeta
        .innerHTML =
        chips.join("");


    elements
        .gameRevenue
        .textContent =
        formatCompactCurrency(
            game.estimated_revenue
        );


    elements
        .gamePrice
        .textContent =

        Number(
            game.price || 0
        ) === 0

            ? "Free"

            : formatCurrency(
                game.price
            );


    elements
        .gameRank
        .textContent =

        "#"
        +
        Number(
            game.revenue_rank_in_year
        ).toLocaleString();


    elements
        .gamePercentile
        .textContent =

        Number(
            game.revenue_percentile_in_year
            || 0
        ).toFixed(1)

        +

        "%";


    elements
        .gameReviews
        .textContent =

        Number(
            game.total_reviews
            || 0
        ).toLocaleString();


    elements
        .gameReviewScore
        .textContent =

        Number(
            game.review_score
            || 0
        ).toFixed(1)

        +

        "%";


    elements
        .gameDescription
        .textContent =

        game.short_description

        ||

        "No short store description is available.";


    elements
        .gameTags
        .innerHTML =

        tags.length

        ?

        tags.map(
            tag =>

                `<span class="tag">
                    ${escapeHtml(tag)}
                </span>`

        ).join("")

        :

        `<span class="tag">
            No tags available
        </span>`;


    const rank =
        Number(
            game.revenue_rank_in_year
            || 0
        );


    const total =
        Number(
            game.qualifying_games_in_year
            || 0
        );


    elements
        .rankCardValue
        .textContent =

        `#${rank.toLocaleString()} of ${total.toLocaleString()}`;


    const topShare =
        total > 0

        ?

        Math.min(
            100,
            100
            *
            rank
            /
            total
        )

        :

        0;


    elements
        .rankCardNote
        .textContent =

        `Approximately the top ${topShare.toFixed(1)}% `
        +
        `of qualifying ${game.release_year} releases. `
        +
        `Dataset percentile: `
        +
        `${Number(
            game.revenue_percentile_in_year || 0
        ).toFixed(1)}th.`;


    elements
        .steamLink
        .href =

        `https://store.steampowered.com/app/${game.appid}`;


    loadMarketingImage(
        game
    );


    elements
        .emptyState
        .classList
        .add("hidden");


    elements
        .caseStudy
        .classList
        .remove("hidden");


    addToHistory(
        game
    );


    window.scrollTo({
        top: 0,
        behavior: "smooth"
    });

}



/* ==========================================================
   HISTORY
   ========================================================== */

function addToHistory(game) {

    const appid =
        Number(
            game.appid
        );


    state
        .shownAppIds
        .add(appid);


    state.history = [

        game,

        ...state.history.filter(
            existing =>
                Number(
                    existing.appid
                )
                !==
                appid
        )

    ].slice(
        0,
        12
    );


    renderHistory();

    updateMatchCount();

}



function renderHistory() {

    if (
        state.history.length
        ===
        0
    ) {

        elements
            .historyPanel
            .classList
            .add("hidden");

        return;

    }


    elements
        .historyPanel
        .classList
        .remove("hidden");


    elements
        .historyList
        .innerHTML =

        state.history.map(

            (game, index) => `

                <div class="history-item">

                    <span class="history-number">

                        ${index + 1}

                    </span>


                    <span class="history-name">

                        ${escapeHtml(
                            game.name
                        )}

                    </span>


                    <span class="history-revenue">

                        ${formatCompactCurrency(
                            game.estimated_revenue
                        )}

                    </span>

                </div>

            `

        ).join("");

}



/* ==========================================================
   RANDOM SUGGESTION
   ========================================================== */

function suggestGame() {

    let candidates =
        filteredGames();


    if (
        candidates.length === 0
        &&
        elements
            .avoidRepeats
            .checked
    ) {

        const allMatches =
            filteredGames({
                respectHistory: false
            });


        if (
            allMatches.length > 0
        ) {

            state
                .shownAppIds
                .clear();


            candidates =
                allMatches;


            showToast(
                "All matching games had been shown. Repeat history reset."
            );

        }

    }


    if (
        candidates.length === 0
    ) {

        showToast(
            "No games match these filters."
        );

        return;

    }


    const selected =

        candidates[
            Math.floor(
                Math.random()
                *
                candidates.length
            )
        ];


    renderGame(
        selected
    );

}



/* ==========================================================
   RESET
   ========================================================== */

function resetFilters() {

    elements
        .yearCheckboxes
        .forEach(
            checkbox => {

                checkbox.checked =
                    true;

            }
        );


    elements.minRevenue.value =
        150000;

    elements.maxRevenue.value =
        800000;


    elements.minPrice.value =
        0;

    elements.maxPrice.value =
        "__MAX_PRICE__";


    elements.minRank.value =
        1;

    elements.maxRank.value =
        "__MAX_RANK__";


    elements.minPercentile.value =
        0;

    elements.maxPercentile.value =
        100;


    elements.minReviews.value =
        20;

    elements.maxReviews.value =
        "";


    elements.minReviewScore.value =
        50;

    elements.maxReviewScore.value =
        100;


    elements.includeTags.value =
        "";

    elements.excludeTags.value =
        "";

    elements.tagSearch.value =
        "";


    elements.avoidRepeats.checked =
        true;


    updateTagUI();

    filterTagBrowser();

    updateMatchCount();


    showToast(
        "Filters reset."
    );

}



/* ==========================================================
   COPY APPID
   ========================================================== */

async function copyAppId() {

    if (
        !state.currentGame
    ) {
        return;
    }


    const text =
        String(
            state.currentGame.appid
        );


    try {

        await navigator
            .clipboard
            .writeText(text);

    }

    catch {

        const area =
            document.createElement(
                "textarea"
            );


        area.value =
            text;


        document
            .body
            .appendChild(
                area
            );


        area.select();


        document.execCommand(
            "copy"
        );


        area.remove();

    }


    showToast(
        `Copied AppID ${text}.`
    );

}



/* ==========================================================
   TOAST
   ========================================================== */

let toastTimer =
    null;


function showToast(message) {

    elements
        .toast
        .textContent =
        message;


    elements
        .toast
        .classList
        .add("visible");


    clearTimeout(
        toastTimer
    );


    toastTimer =
        setTimeout(

            () => {

                elements
                    .toast
                    .classList
                    .remove(
                        "visible"
                    );

            },

            2400

        );

}



/* ==========================================================
   FILTER LISTENERS
   ========================================================== */

const filterInputs = [

    ...elements.yearCheckboxes,

    elements.minRevenue,

    elements.maxRevenue,

    elements.minPrice,

    elements.maxPrice,

    elements.minRank,

    elements.maxRank,

    elements.minPercentile,

    elements.maxPercentile,

    elements.minReviews,

    elements.maxReviews,

    elements.minReviewScore,

    elements.maxReviewScore,

    elements.includeTags,

    elements.excludeTags,

    elements.avoidRepeats

];


filterInputs.forEach(

    element => {

        element.addEventListener(

            "input",

            () => {

                if (
                    element
                    ===
                    elements.includeTags
                ) {

                    updateTagUI();

                }


                updateMatchCount();

            }

        );


        element.addEventListener(

            "change",

            () => {

                if (
                    element
                    ===
                    elements.includeTags
                ) {

                    updateTagUI();

                }


                updateMatchCount();

            }

        );

    }

);



/* ==========================================================
   TAG LISTENERS
   ========================================================== */

elements
    .availableTagButtons
    .forEach(

        button => {

            button.addEventListener(

                "click",

                () => {

                    addIncludedTag(
                        button.dataset.tag
                    );

                }

            );

        }

    );


elements
    .tagSearch
    .addEventListener(

        "input",

        filterTagBrowser

    );



/* ==========================================================
   LOOKUP LISTENERS
   ========================================================== */

elements
    .gameLookup
    .addEventListener(

        "input",

        renderLookupResults

    );


elements
    .gameLookup
    .addEventListener(

        "keydown",

        event => {

            if (
                event.key
                ===
                "Enter"
            ) {

                event.preventDefault();

                loadSpecificGame();

            }

        }

    );


elements
    .loadGameButton
    .addEventListener(

        "click",

        loadSpecificGame

    );


document.addEventListener(

    "click",

    event => {

        if (
            !event.target.closest(
                ".lookup-wrap"
            )
        ) {

            elements
                .lookupResults
                .classList
                .add("hidden");

        }

    }

);



/* ==========================================================
   MAIN BUTTON LISTENERS
   ========================================================== */

elements
    .suggestButton
    .addEventListener(

        "click",

        suggestGame

    );


elements
    .anotherButton
    .addEventListener(

        "click",

        suggestGame

    );


elements
    .resetButton
    .addEventListener(

        "click",

        resetFilters

    );


elements
    .copyAppIdButton
    .addEventListener(

        "click",

        copyAppId

    );


elements
    .clearHistoryButton
    .addEventListener(

        "click",

        () => {

            state
                .shownAppIds
                .clear();


            state.history =
                [];


            elements
                .historyList
                .innerHTML =
                "";


            elements
                .historyPanel
                .classList
                .add("hidden");


            updateMatchCount();


            showToast(
                "Session history cleared."
            );

        }

    );



/* ==========================================================
   INITIALIZE
   ========================================================== */

updateTagUI();

filterTagBrowser();

updateMatchCount();


</script>


</body>

</html>
"""


# ============================================================
# REPLACE TEMPLATE PLACEHOLDERS
# ============================================================

html_document = (
    html_document
    .replace(
        "__GAME_COUNT__",
        f"{len(games_data):,}"
    )
    .replace(
        "__YEAR_CHECKBOXES__",
        year_checkbox_html
    )
    .replace(
        "__TAG_COUNT__",
        f"{len(all_tags):,}"
    )
    .replace(
        "__TAG_BUTTONS__",
        tag_buttons_html
    )
    .replace(
        "__MAX_PRICE__",
        str(
            max(
                100,
                int(maximum_price) + 1
            )
        )
    )
    .replace(
        "__MAX_RANK__",
        str(maximum_rank)
    )
    .replace(
        "__GAME_DATA__",
        embedded_json
    )
)


# ============================================================
# WRITE HTML
#
# Using chunked binary output avoids the Windows issue you
# encountered with the ~10 MB generated HTML.
# ============================================================

html_bytes = html_document.encode(
    "utf-8",
    errors="replace"
)

CHUNK_SIZE = 1_000_000

with open(
    HTML_OUTPUT_FILE,
    "wb"
) as file:

    for start in range(
        0,
        len(html_bytes),
        CHUNK_SIZE
    ):

        file.write(
            html_bytes[
                start:
                start + CHUNK_SIZE
            ]
        )


# ============================================================
# VERIFY OUTPUT
# ============================================================

file_size_mb = (
    HTML_OUTPUT_FILE.stat().st_size
    / 1_000_000
)

print(
    "Created Steam suggester:"
)

print(
    HTML_OUTPUT_FILE.resolve()
)

print()

print(
    f"Embedded games: {len(games_data):,}"
)

print(
    f"Available tags: {len(all_tags):,}"
)

print(
    f"File size: {file_size_mb:.2f} MB"
)

Created Steam suggester:
D:\Workstation\python\steam-analysis\steam-game-suggester\steam_game_suggester.html

Embedded games: 15,521
Available tags: 443
File size: 11.17 MB


In [3]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("APP_OUTPUT_DIR:", APP_OUTPUT_DIR)
print("HTML_OUTPUT_FILE:", HTML_OUTPUT_FILE)
print("Resolved:", HTML_OUTPUT_FILE.resolve())
print("Exists:", APP_OUTPUT_DIR.exists())
print("HTML characters:", len(html_document))

PROJECT_ROOT: d:\Workstation\python\steam-analysis
APP_OUTPUT_DIR: d:\Workstation\python\steam-analysis\steam-game-suggester
HTML_OUTPUT_FILE: d:\Workstation\python\steam-analysis\steam-game-suggester\steam_game_suggester.html
Resolved: D:\Workstation\python\steam-analysis\steam-game-suggester\steam_game_suggester.html
Exists: True
HTML characters: 11092178
